In [ ]:
import evaluate

In [ ]:
# 评估函数
evaluate.list_evaluation_modules()

In [ ]:
evaluate.list_evaluation_modules(include_community=True, with_details=True)

In [ ]:
accuracy = evaluate.load("accuracy")

In [ ]:
accuracy.description

In [ ]:
print(accuracy.description)

In [ ]:
print(accuracy.inputs_description)

In [ ]:
results = accuracy.compute(references=[1, 1, 2, 0, 1, 2], predictions=[0, 1, 1, 2, 1, 0])
print(results)

In [ ]:
accuracy

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

In [ ]:
dataset = load_dataset("csv", data_files="./ChnSentiCorp_htl_all.csv", split="train")
dataset = dataset.filter(lambda x: x["review"] is not None)
dataset

In [19]:
datasets = dataset.train_test_split(test_size=0.1)
datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'review'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['label', 'review'],
        num_rows: 777
    })
})

In [20]:
import torch

tokenizer = AutoTokenizer.from_pretrained("hfl/rbt3")


def process_func(example):
    model_inputs = tokenizer(example["review"], max_length=128, truncation=True)
    model_inputs["labels"] = example["label"]
    return model_inputs


tokenized_dataset = datasets.map(process_func, batched=True, remove_columns=datasets["train"].column_names)
tokenized_dataset

Map: 100%|██████████| 777/777 [00:00<00:00, 5713.37 examples/s]


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 6988
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 777
    })
})

In [21]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

trainloader = DataLoader(tokenized_dataset["train"], batch_size=32,
                         collate_fn=DataCollatorWithPadding(tokenizer=tokenizer), shuffle=True)
validloader = DataLoader(tokenized_dataset["test"], batch_size=64,
                         collate_fn=DataCollatorWithPadding(tokenizer=tokenizer), shuffle=True)

In [22]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('hfl/rbt3')
if torch.cuda.is_available():
    model = model.cuda()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at hfl/rbt3 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [24]:
import evaluate

clf_metric = evaluate.combine(["accuracy", "f1", "precision", "recall"])

In [26]:
def evaluate():
    model.eval()
    with torch.inference_mode():
        for batch in validloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            output = model(**batch)
            pred = torch.argmax(output.logits, dim=-1)
            clf_metric.add_batch(predictions=pred.long(), references=batch["labels"].long())
    return clf_metric.compute()


def train(epoch=3, log_step=100):
    global_step = 0
    for ep in range(epoch):
        model.train()
        for batch in trainloader:
            if torch.cuda.is_available():
                batch = {k: v.cuda() for k, v in batch.items()}
            optimizer.zero_grad()
            output = model(**batch)
            output.loss.backward()
            optimizer.step()
            if global_step % log_step == 0:
                print(f"ep: {ep}, global_step: {global_step}, loss: {output.loss.item()}")
            global_step += 1
        clf = evaluate()
        print(f"ep: {ep},clf: {clf}")


In [27]:
train()

ep: 0, global_step: 0, loss: 0.7277461290359497
ep: 0, global_step: 100, loss: 0.36555442214012146
ep: 0, global_step: 200, loss: 0.15146321058273315
ep: 0,clf: {'accuracy': 0.888030888030888, 'f1': 0.9193697868396663, 'precision': 0.9117647058823529, 'recall': 0.9271028037383178}
ep: 1, global_step: 300, loss: 0.4250948131084442
ep: 1, global_step: 400, loss: 0.12001948803663254
ep: 1,clf: {'accuracy': 0.8931788931788932, 'f1': 0.9233610341643582, 'precision': 0.9124087591240876, 'recall': 0.9345794392523364}
ep: 2, global_step: 500, loss: 0.35750699043273926
ep: 2, global_step: 600, loss: 0.13407287001609802
ep: 2,clf: {'accuracy': 0.9009009009009009, 'f1': 0.9283720930232559, 'precision': 0.924074074074074, 'recall': 0.9327102803738317}
